In [9]:
%pip install yagmail
%pip install langchain_google_genai langgraph langchain 
%pip install imap-tools
%pip install google-adk

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
from dotenv import load_dotenv
import os

load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")
serp_api_key=os.getenv("SERP_API_KEY")
app_password=os.getenv("APP_PASSWORD")
google_generative_ai_api=os.getenv("Google_Generative_AI_API")
email=os.getenv("EMAIL")


In [ ]:

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from pydantic import SecretStr
import yagmail

llm2 = ChatGroq(            #new Agent with Groq LLM
    model="openai/gpt-oss-120b",
    temperature=0.1,            #temperature higher means more creative and lower means strict in output 
    max_retries=2,
    api_key=SecretStr(groq_api_key) if groq_api_key is not None else None
)

llm=ChatGoogleGenerativeAI(
    api_key=SecretStr(google_generative_ai_api) if google_generative_ai_api is not None else None,
    temperature=0.3,        #More on Strict side, less randomness in the output.
    model="gemini-flash-lite-latest",
    max_tokens=None,
    max_retries=2,
    timeout=None
)
sys_prompt = """Your Name is Frankenstein — a loyal right-hand assistant and butler. Highly intelligent, fiercely protective, and executes every command to perfection.
You are an Email Assistant powered by Google Gemini.
You assist Muneeb with reading, searching, drafting, and sending emails.

Rules:
- Always draft emails using draft_email_tool first unless Muneeb explicitly orders you to send directly.
- Use send_email_tool for standard outgoing emails and send_email_with_attachment_tool whenever a file attachment path is provided.
- Research with serpapi_search only when outside context or facts are needed.
- Use get_current_datetime_tool for any date or time references in messages.
- Read and summarize email threads using summarize_thread_tool before drafting a reply if thread context exists.
- Keep all communication concise, professional, and devoid of unnecessary filler.
- Maintain an ultra-polite, highly loyal, and impeccably sharp tone."""

yag = yagmail.SMTP(email,password=app_password)
# Simple contact dictionary
CONTACTS = {
    "ahmed": "ahmed@mailinator.com",
    "ali": "ali@mailinator.com",
    "muneeb": "muneeb@mailinator.com",
    "imtinan": "imtinan@mailinator.com",
    "ryuk":"ryuk@mailinator.com"
}

In [ ]:
from zoneinfo import ZoneInfo
from langchain_core.tools import tool
from serpapi import GoogleSearch
from langgraph.checkpoint.memory import InMemorySaver
from datetime import datetime
from imap_tools import MailBox, AND
from typing import Any

# @tool decorator and a proper triple-quoted docstring to define the Tool's Functionality.
@tool
def send_email_tool(recipient: str, subject: str, contents: str) -> str:
    """Sends an email to the recipient with the specified subject and contents."""
    try:
        yag.send(
            to=recipient,
            subject=subject,
            contents=contents
        )
        return "Email sent!"
    except Exception as e:
        return f"Error sending email: {e}"
    
#lets add Serp Search in it 
@tool
def serpapi_search(query: str) -> list[dict[str, Any]]:
    """Searches for a query using the SerpAPI on Google."""
    try:
        # Check API key before initializing search
        if not serp_api_key:
            return [{"error": "serp_api_key is missing or empty"}]
        
        params = {
            "q": query,
            "hl": "en",
            "gl": "us",
            "api_key": serp_api_key
        }
        
        # Perform the external call inside the try block
        search = GoogleSearch(params)
        results = search.get_dict()

        if "organic_results" in results and results["organic_results"]:
            return [
                {
                    "title": r["title"],
                    "link": r["link"],
                    "snippet": r.get("snippet", "")
                }
                for r in results["organic_results"][:5]
            ]
            
        return [{"error": "No results found"}]
        
    except Exception as e:
        # Catches network errors, API failures, or missing variables
        return [{"error": f"Failed to perform search: {e}"}]

@tool
def read_inbox_tool(count: int = 5) -> list:
    """Reads the most recent emails from the inbox using imap-tools (simpler alternative to raw imaplib)."""
    try:
        if not email or not app_password:
            return [{"error": "EMAIL or APP_PASSWORD environment variable is not set"}]
        results = []
        with MailBox("imap.gmail.com").login(email, app_password) as mailbox:
            for msg in mailbox.fetch(AND(all=True), reverse=True, limit=count):
                results.append({
                    "from": msg.from_,
                    "subject": msg.subject,
                    "snippet": (msg.text or msg.html or "")[:200]
                })
        return results
    except Exception as e:
        return [{"error": f"Failed to read inbox: {e}"}]

@tool
def draft_email_tool(recipient: str, subject: str, contents: str) -> dict:
    """Creates a draft email (does NOT send it) so it can be reviewed before sending."""
    return {
        "to": recipient,
        "subject": subject,
        "body": contents,
        "status": "draft - not sent yet"
    }

@tool
def send_email_with_attachment_tool(recipient: str, subject: str, contents: str, file_path: str) -> str:
    """Sends an email with a single file attached, using yagmail."""
    try:
        yag.send(
            to=recipient,
            subject=subject,
            contents=contents,
            attachments=file_path
        )
        return "Email with attachment sent!"
    except Exception as e:
        return f"Error sending email with attachment: {e}"

@tool
def get_current_datetime_tool(timezone: str = "Asia/Karachi") -> str:
    """Returns the current date and time, so the agent doesn't have to guess it."""
    try:
        now = datetime.now(ZoneInfo(timezone))
        return now.strftime("%A, %d %B %Y, %I:%M %p")   
    except Exception as e:
        return f"Error getting current datetime: {e}"
from langchain_core.tools import tool

@tool
def summarize_thread_tool(thread_text: str) -> str:
    """Summarizes a long email thread into a short paragraph using the LLM."""
    try:
        prompt = (
            "Summarize this email thread in 3-4 short sentences, "
            f"focusing on key points and any action items:\n\n{thread_text}"
        )
        response = llm.invoke(prompt)
        
        # Ensure the content is returned as a string regardless of LLM response object
        return str(response.content) if hasattr(response, "content") else str(response)
    except Exception as e:
        return f"Error summarizing thread: {e}"


@tool
def contact_lookup_tool(name: str) -> str:
    """Finds an email address by contact name from the contact book."""
    clean_name = name.strip().lower()
    if clean_name in CONTACTS:
        return CONTACTS[clean_name]
    return f"Contact '{name}' not found. Available contacts: {', '.join(CONTACTS.keys())}"

tools_registry = [
    draft_email_tool,
    send_email_tool,
    send_email_with_attachment_tool,
    serpapi_search,
    read_inbox_tool,
    get_current_datetime_tool,
    summarize_thread_tool,
    contact_lookup_tool
]
memory = InMemorySaver()
# Create your agent


In [ ]:
from langchain.agents import create_agent
graph = create_agent(
    name="Email Writing Agent",
    model=llm,
    tools=tools_registry,
    system_prompt = sys_prompt,
    checkpointer=memory
)


In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig

contents = input("Please tell me who do you wanna send email to and what do you want to say in the email?")

# Explicitly type config as RunnableConfig
config: RunnableConfig = {
    "configurable": {
        "thread_id": "email_agent_session_123"
    }
}

for chunk in graph.stream({"messages": [HumanMessage(content=contents)]
}, config=config, stream_mode="updates"): 
    print(chunk)

{'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'serpapi_search', 'arguments': '{"query": "best fuel efficient used sedans crossovers cars pakistan price"}'}, '__gemini_function_call_thought_signatures__': {'5sCky7bk': 'EjQKMgERTTIPdDzhjF/HCRZx/nJbEZZAVKO8tEWqOcTNeveC/yh/KSOdTJ9Wr3F9I83Npzvr'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, name='Email Writing Agent', id='lc_run--019fbcea-c12e-7543-a390-016ba7a61f06-0', tool_calls=[{'name': 'serpapi_search', 'args': {'query': 'best fuel efficient used sedans crossovers cars pakistan price'}, 'id': '5sCky7bk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 4645, 'output_tokens': 27, 'total_tokens': 4672, 'input_token_details': {'cache_read': 0}})]}}
{'tools': {'messages': [ToolMessage(content='[{"title": "What is a good crossover car to buy in Pakistan within ...", "link"